# Notebook for running the winning model




In [1]:
import json
from pathlib import Path

import pandas as pd

import processing.acled_events_processing as acled
from models.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

REPORTS_DIR = Path("evaluation/model_reports")
reports_dir = Path(REPORTS_DIR)
reports_dir.mkdir(parents=True, exist_ok=True)

In [2]:
def summarise(label, subset):
    n_true_pos = subset["y_true"].sum()
    n_caught = subset[(subset["y_true"] == 1) & (subset["y_pred"] == 1)].shape[0]
    recall = n_caught / n_true_pos if n_true_pos else float("nan")
    n_pred_pos = subset["y_pred"].sum()
    precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
    print(
        f"{label}: {len(subset)} rows, {n_true_pos} true escalations, "
        f"{n_caught} caught -> recall={recall:.3f}, precision={precision:.3f}"
    )
    return {
        "n_rows": len(subset),
        "n_true_pos": int(n_true_pos),
        "n_caught": n_caught,
        "recall": recall,
        "precision": precision,
    }

In [3]:
def run_model(config, params):
    data_sources = [
        src
        for src, include in zip(
            ["food", "rain", "text"],
            [
                config["include_food"],
                config["include_rain"],
                config["include_text"],
            ],
        )
        if include
    ]

    model_data, predictor_cols = get_clean_combined_data(
        data_sources=data_sources,
        k=config["k"],
        event_col=config["event_col"],
        conflict_only_embeddings=config["conflict_only"],
    )

    final_params = {
        **params,
        "k": config["k"],
        "event_col": config["event_col"],
        "n_splits": config["n_splits"],
        "use_pca": config["use_pca"],
    }

    results, best_params, shap_importance, onset_predictions = train_evaluate_model(
        model_data,
        predictor_cols,
        final_params,
        best_params=True,  # skip RandomizedSearchCV
        use_pca=config["use_pca"],
        compute_shap=True,
        shap_sample_size=2000,
        return_onset_predictions=True,
    )
    return results, best_params, shap_importance, onset_predictions

In [4]:
def model_report(label, config, params):
    """Runs a model and prints the full report, same as before, but also
    returns a flat dict of key metrics so multiple models can be compared
    in a table afterwards without re-typing numbers by hand.
    """
    results, best_params, shap_importance, onset_predictions = run_model(config, params)
    print("========MODEL REPORT========")
    print(f"---Model: {label}\n")
    print("---Results\n")
    print(results)
    print("---Best Params\n")
    print(best_params)
    print("---SHAP Importance\n")
    print(shap_importance)

    onset_predictions["year_month"] = onset_predictions["year_month"].astype(str)
    war_outbreak = "2023-04"
    print("Pre and post war:")
    pre_war = onset_predictions[onset_predictions["year_month"] < war_outbreak]
    post_war = onset_predictions[onset_predictions["year_month"] >= war_outbreak]

    pre_war_summary = summarise("Pre-war  (Jan-Mar 2023)", pre_war)
    post_war_summary = summarise("Post-war (Apr-Dec 2023)", post_war)

    key_regions = [
        "Khartoum",
        "North Darfur",
        "South Darfur",
        "West Darfur",
        "Central Darfur",
        "East Darfur",
        "West Kordofan",
        "South Kordofan",
    ]

    print("-----Key war-affected regions\n")
    key_region_rows = onset_predictions[onset_predictions["region"].isin(key_regions)]
    key_regions_summary = summarise("Key regions (all onset months)", key_region_rows)

    khartoum_rows = onset_predictions[onset_predictions["region"] == "Khartoum"]
    khartoum_summary = summarise("  Khartoum", khartoum_rows)

    for region in key_regions:
        if region == "Khartoum":
            continue
        region_rows = onset_predictions[onset_predictions["region"] == region]
        if region_rows["y_true"].sum() > 0:
            summarise(f"  {region}", region_rows)

    comparison_row = {
        "model": label,
        "onset_aupr": float(results["onset_aupr"]),
        "active_aupr": float(results["active_aupr"]),
        "pre_war_recall": pre_war_summary["recall"],
        "post_war_recall": post_war_summary["recall"],
        "khartoum_recall": khartoum_summary["recall"],
        "key_regions_recall": key_regions_summary["recall"],
    }

    return results, best_params, shap_importance, onset_predictions, comparison_row

In [5]:
def save_model_report(label, results, best_params, shap_importance, onset_predictions):
    label_formatted = label.replace(" ", "_").replace("(", "").replace(")","")
    
    with open(REPORTS_DIR / f"{label_formatted}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    
    with open(REPORTS_DIR / f"{label_formatted}_best_params.json", "w") as f:
        json.dump(best_params, f, indent=2)
        
    shap_importance.to_csv(REPORTS_DIR / f"{label_formatted}_shap.csv", index=False)
    onset_predictions.to_csv(REPORTS_DIR / f"{label_formatted}_onset_predictions.csv", index=False)

In [6]:
def open_model_report(label):
    label_formatted = label.replace(" ", "_").replace("(", "").replace(")","")

    with open(REPORTS_DIR / f"{label_formatted}_results.json") as f:
            results = json.load(f)
    with open(REPORTS_DIR / f"{label_formatted}_best_params.json") as f:
        best_params = json.load(f)
    shap_importance = pd.read_csv(REPORTS_DIR / f"{label_formatted}_shap.csv")
    onset_predictions = pd.read_csv(REPORTS_DIR / f"{label_formatted}_onset_predictions.csv")
    
    return results, best_params, shap_importance, onset_predictions

## Model A - Structural only

In [18]:
model_a_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": False,
    "conflict_only": None,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": False,
}

# (acled_sub_food_rain_threshold_change_1.75_5, onset_aupr=0.3336, active_aupr=0.2025)
model_a_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.01,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 1,
    "colsample_bylevel": 1.0,
}

In [19]:
results_a, best_params_a, shap_a, onset_preds_a, row_a = model_report(
    "Model A", model_a_config, model_a_xgb_params
)
save_model_report("Model A", results_a, best_params_a, shap_a, onset_preds_a)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Name Mapping:Renamed region 'Eastern Darfur' to 'East Darfur' (manual override)
INFO:Name Mapping:Renamed region 'Al Gezira' to 'Al Jazirah' (manual override)
INFO:Name Mapping:Renamed region 'Nile' to 'River Nile' (manual override)
INFO:Data preparation:Food prices data processed.
INFO:Name Mapping:Creating a PCODE map for rainfall data. This may take a few seconds...
INFO:HDXClient:Reading local file data/hdx/admin_boundaries/hdx_sdn_admin1.geojson
INFO:HDXClient:Reading local file: data/hdx/hdx_sdn_rainfall_subnat_full.csv
INFO:Name Mapping:Renamed region 'Aj Jazirah' to 'Al Jazirah' (manual override)
INFO:Rainfal

--- Fold 1 ---
Train window: 2018-01 to 2018-10 (180 rows)
Test window:  2018-11 to 2019-08 (180 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (360 rows)
Test window:  2019-09 to 2020-06 (180 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (540 rows)
Test window:  2020-07 to 2021-04 (180 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (720 rows)
Test window:  2021-05 to 2022-02 (180 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (900 rows)
Test window:  2022-03 to 2022-12 (180 rows)
------------------------------
========MODEL REPORT========
---Model: Model A

---Results

{'optimal_threshold': '0.4255', 'n_predictors': 31, 'onset_aupr': '0.3336', 'onset_precision_class1': '0.2736', 'onset_recall_class1': '0.5918', 'onset_f1_class1': '0.3742', 'active_aupr': '0.2025', 'active_precision_class1': '0.1860', 'active_recall_class1': '0.5517', 

## Model B

### Best model (conflict-only text)

In [ ]:
# --- Model B: numeric + food + rain + text (matched to Model A) ---
model_b_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": True,
    "conflict_only": True,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": True,
}

# (acled_sub_food_rain_text_conflict_pca_1.75_5, onset_aupr=0.3941, active_aupr=0.2155)
model_b_xgb_params = {
    "max_depth": 7,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 3,
    "learning_rate": 0.01,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 2.0,
    "reg_lambda": 10,
    "colsample_bylevel": 0.8,
}

In [10]:
(results_b_conflict,
    best_params_b_conflict,
    shap_b_conflict,
    onset_preds_b_conflict,
    row_b_conflict,
) = model_report("Model B (conflict-only text)", model_b_config, model_b_xgb_params)

save_model_report("Model B (conflict-only text)", results_b_conflict,
    best_params_b_conflict,
    shap_b_conflict,
    onset_preds_b_conflict,)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Name Mapping:Renamed region 'Eastern Darfur' to 'East Darfur' (manual override)
INFO:Name Mapping:Renamed region 'Al Gezira' to 'Al Jazirah' (manual override)
INFO:Name Mapping:Renamed region 'Nile' to 'River Nile' (manual override)
INFO:Data preparation:Food prices data processed.
INFO:Name Mapping:Creating a PCODE map for rainfall data. This may take a few seconds...
INFO:HDXClient:Reading local file data/hdx/admin_boundaries/hdx_sdn_admin1.geojson
INFO:HDXClient:Reading local file: data/hdx/hdx_sdn_rainfall_subnat_full.csv
INFO:Name Mapping:Renamed region 'Aj Jazirah' to 'Al Jazirah' (manual override)
INFO:Rainfal

--- Fold 1 ---
Train window: 2018-01 to 2018-10 (180 rows)
Test window:  2018-11 to 2019-08 (180 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (360 rows)
Test window:  2019-09 to 2020-06 (180 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (540 rows)
Test window:  2020-07 to 2021-04 (180 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (720 rows)
Test window:  2021-05 to 2022-02 (180 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (900 rows)
Test window:  2022-03 to 2022-12 (180 rows)
------------------------------
========MODEL REPORT========
---Model: Model B (conflict-only text)

---Results

{'optimal_threshold': '0.4370', 'n_predictors': 56, 'onset_aupr': '0.3941', 'onset_precision_class1': '0.3158', 'onset_recall_class1': '0.3673', 'onset_f1_class1': '0.3396', 'active_aupr': '0.2155', 'active_precision_class1': '0.1746', 'active_recal

### Comparison (all text)

In [22]:
# --- Model B: numeric + food + rain + text (matched to Model A) ---
model_b_all_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": True,
    "conflict_only": False,
    "k": 1.75,
    "event_col": "event_type",  # Broad events work best for narrative context
    "n_splits": 5,
    "use_pca": False,           # Raw embeddings retain the most predictive signal
}

model_b_all_xgb_params = {
    "max_depth": 5,
    "min_child_weight": 1,
    "max_delta_step": 1,
    "gamma": 5,
    "learning_rate": 0.01,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0,
    "reg_lambda": 5,
    "colsample_bylevel": 1.0,
}

In [23]:
results_b_all, best_params_b_all, shap_b_all, onset_preds_b_all, row_b_all = (
    model_report("Model B (all-event text)", model_b_all_config, model_b_all_xgb_params)
)
save_model_report("Model B (all-event text)",results_b_all, best_params_b_all, shap_b_all, onset_preds_b_all)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Name Mapping:Renamed region 'Eastern Darfur' to 'East Darfur' (manual override)
INFO:Name Mapping:Renamed region 'Al Gezira' to 'Al Jazirah' (manual override)
INFO:Name Mapping:Renamed region 'Nile' to 'River Nile' (manual override)
INFO:Data preparation:Food prices data processed.
INFO:Name Mapping:Creating a PCODE map for rainfall data. This may take a few seconds...
INFO:HDXClient:Reading local file data/hdx/admin_boundaries/hdx_sdn_admin1.geojson
INFO:HDXClient:Reading local file: data/hdx/hdx_sdn_rainfall_subnat_full.csv
INFO:Name Mapping:Renamed region 'Aj Jazirah' to 'Al Jazirah' (manual override)
INFO:Rainfall Pr

--- Fold 1 ---
Train window: 2018-01 to 2018-10 (180 rows)
Test window:  2018-11 to 2019-08 (180 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (360 rows)
Test window:  2019-09 to 2020-06 (180 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (540 rows)
Test window:  2020-07 to 2021-04 (180 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (720 rows)
Test window:  2021-05 to 2022-02 (180 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (900 rows)
Test window:  2022-03 to 2022-12 (180 rows)
------------------------------
========MODEL REPORT========
---Model: Model B (all-event text)

---Results

{'optimal_threshold': '0.4151', 'n_predictors': 782, 'onset_aupr': '0.3624', 'onset_precision_class1': '0.3010', 'onset_recall_class1': '0.6327', 'onset_f1_class1': '0.4079', 'active_aupr': '0.1733', 'active_precision_class1': '0.1436', 'active_recall_c

In [25]:
corpus_comparison = pd.DataFrame([row_a, row_b_conflict, row_b_all]).set_index("model")
corpus_comparison.loc["Difference (all-text - conflict-only)"] = (
    corpus_comparison.loc["Model B (all-event text)"]
    - corpus_comparison.loc["Model B (conflict-only text)"]
)
corpus_comparison.round(3)

,onset_aupr,active_aupr,pre_war_recall,post_war_recall,khartoum_recall,key_regions_recall
model,,,,,,
Model A,0.334,0.202,0.625,0.585,0.2,0.556
Model B (conflict-only text),0.394,0.216,0.250,0.390,0.0,0.259
Model B (all-event text),0.362,0.173,0.375,0.683,0.4,0.704
Difference (all-text - conflict-only),-0.032,-0.042,0.125,0.293,0.4,0.444


In [14]:
_, _, raw_df = acled.get_clean_data(k=1.75, event_col="sub_event_type")

if not isinstance(raw_df["year_month"].dtype, pd.PeriodDtype):
    raw_df["year_month"] = pd.to_datetime(raw_df["year_month"]).dt.to_period("M")

pre_war = raw_df[
    (raw_df["year_month"] >= "2023-01") & (raw_df["year_month"] <= "2023-03")
]

key_regions = [
    "Khartoum",
    "North Darfur",
    "South Darfur",
    "West Darfur",
    "Central Darfur",
    "East Darfur",
    "West Kordofan",
    "South Kordofan",
]

summary = (
    pre_war[pre_war["admin1"].isin(key_regions)]
    .groupby("admin1")
    .agg(total_events=("conflict", "size"), conflict_events=("conflict", "sum"))
)
summary["pct_conflict"] = (
    summary["conflict_events"] / summary["total_events"] * 100
).round(1)
summary.sort_values("conflict_events")

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.


,total_events,conflict_events,pct_conflict
admin1,,,
East Darfur,10,4,40.0
West Kordofan,15,4,26.7
South Kordofan,30,15,50.0
West Darfur,32,22,68.8
Khartoum,220,25,11.4
South Darfur,51,27,52.9
Central Darfur,38,29,76.3
North Darfur,85,44,51.8


In [15]:
import pandas as pd

from utils.constants import ACTIVE_END_DATE, ACTIVE_START_DATE

active_prevalence_records = []
for k_test in [1.75]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[],
        k=k_test,
        event_col="sub_event_type",
        conflict_only_embeddings=True,
    )
    active_slice = model_data[
        (model_data["year_month"] >= pd.Period(ACTIVE_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ACTIVE_END_DATE, freq="M"))
    ]
    n_total = len(active_slice)
    n_escalations = int(active_slice["target_escalation"].sum())
    active_prevalence_records.append(
        {
            "k": k_test,
            "active_prevalence_percentage": round(n_escalations / n_total * 100, 1),
            "n_escalations": n_escalations,
            "n_active_rows": n_total,
        }
    )

active_prevalence_df = pd.DataFrame(active_prevalence_records).set_index("k")
print(active_prevalence_df)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.


      active_prevalence_percentage  n_escalations  n_active_rows
k                                                               
1.75                          13.4             58            432


### AUPR relative to baseline

Raw AUPR values at k=1.75 are lower than at k=1.0 (Model A onset: 0.3379 vs 0.3213 previously;
active: 0.2242 vs 0.3693 previously), which could look like a regression. However, AUPR's
no-skill baseline equals the positive-class prevalence, which is itself lower at the stricter
k=1.75 threshold (22.7% onset, 16.2% active, vs 30.6% onset at k=1.0). Expressed as a lift over
that baseline, Model A achieves 1.49x on onset and 1.38x on active, a genuine, meaningful
improvement over chance, and the onset lift is comparable to or better than what was observed at
k=1.0. The apparent decline in raw AUPR is therefore largely explained by evaluating against a
stricter, more legitimate target, not by weaker model performance. [Model B's equivalent lift
figures are not yet available and require re-running with local embeddings access.]


### Regional performance

Onset-window recall varies substantially by region. Of the key regions in the onset months (Jan-Mar 2023), Model A caught 15 of the 27 true escalations. Notably, Model A only caught only 1 of 5 true escalations in Khartoum (recall 0.200) - the capital and the site of the actual April 2023 outbreak. Recall was better in East Darfur, catching 3 of 3 true esclations, and West Kordofan (4 of 4). 

Model B (which includes conflict-only text) has very poor regional performance, catching 0 esclations in Khatoum and only 7 of the 27 true escalations caught. 

The interesting thing is that at the regional level during the onset period, Model B with all text performs better in terms of recall. It manages to catch 23 of the 27 true escalations. Looking at the data for Khatoum this is likely because of the 220 events in the onset months, only 11.4% were conflict events, the others were things like strategic developments. 

These finding are of a very small sample size (1-5 true esclations per region) so can only be taken as tenative. Comparing Model B all text vs conflict-only text, all text 
